In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
from ucimlrepo import fetch_ucirepo
import os
from pathlib import Path

# --- KAGGLE AUTHENTICATION ---
# Credentials are loaded from ~/.kaggle/kaggle.json or environment variables.
# See .env.example for setup instructions.
if not os.environ.get("KAGGLE_USERNAME"):
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    if not kaggle_json.exists():
        raise FileNotFoundError(
            "Kaggle credentials not found. Place kaggle.json in ~/.kaggle/ "
            "or set KAGGLE_USERNAME and KAGGLE_KEY environment variables. "
            "See .env.example for details."
        )

# We must import kaggle AFTER credentials are available
import kaggle 
# ---------------------------------

# Add the src folder to the path — search upward for src/config.py
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src" / "config.py").exists():
    pass  # Already at project root
elif (PROJECT_ROOT.parent / "src" / "config.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise FileNotFoundError("Cannot locate src/config.py. Run this notebook from the project root or notebooks/ directory.")
import sys
sys.path.insert(0, str(PROJECT_ROOT))
import src.config as config

# Setup visualization style
sns.set_theme(style="whitegrid")
print("Libraries imported and Kaggle authenticated successfully!")

2026-04-09 14:59:48 - src.config - INFO - Project configuration loaded and directories verified.


Libraries imported and Kaggle authenticated successfully!


In [ ]:
print("Downloading UCI Datasets...")

# Dataset 1: Student Performance (Math & Portuguese)
student_perf = fetch_ucirepo(id=320)
df_student_math = student_perf.data.original
# Save to raw data folder
df_student_math.to_csv(config.RAW_DATA_DIR / "uci_student_performance.csv", index=False)

# Dataset 2: Dropout & Academic Success
dropout_success = fetch_ucirepo(id=697)
df_dropout = dropout_success.data.original
df_dropout.to_csv(config.RAW_DATA_DIR / "uci_dropout_success.csv", index=False)

print("UCI Datasets downloaded and saved to /data/raw/")

In [ ]:
print("Downloading Kaggle Dataset...")
dataset_name = "rabieelkharoua/students-performance-dataset"
kaggle_download_path = config.RAW_DATA_DIR

# Download using the Kaggle API
kaggle.api.dataset_download_files(
    dataset_name, 
    path=kaggle_download_path, 
    unzip=True
)

# Robust file discovery (handles name variations)
csv_matches = list(kaggle_download_path.glob("Student*performance*.csv"))
if len(csv_matches) == 0:
    csv_matches = list(kaggle_download_path.glob("*.csv"))
if len(csv_matches) != 1:
    raise FileNotFoundError(
        f"Expected 1 Kaggle CSV, found {len(csv_matches)}: {csv_matches}"
    )
kaggle_file = csv_matches[0]
df_kaggle = pd.read_csv(kaggle_file)
kaggle_file.rename(config.RAW_DATA_DIR / "kaggle_student_performance.csv")

print("Kaggle dataset downloaded and unzipped to /data/raw/")

In [ ]:
datasets = {
    "UCI_Student_Performance": df_student_math,
    "UCI_Dropout_Success": df_dropout,
    "Kaggle_Student_Performance": df_kaggle
}

for name, df in datasets.items():
    print(f"Generating profile for {name}...")
    profile = ProfileReport(df, title=f"{name} Profiling Report", minimal=True)
    # Save reports to the paper folder or root for easy access
    report_path = config.PROJECT_ROOT / "paper" / f"{name}_EDA_Report.html"
    profile.to_file(report_path)

print("All HTML EDA reports generated successfully in the /paper directory!")

In [ ]:
def plot_class_distribution(df, target_col, dataset_name):
    """Plots the class distribution to check for imbalance."""
    if target_col in df.columns:
        plt.figure(figsize=(8, 4))
        sns.countplot(data=df, x=target_col, palette="viridis")
        plt.title(f"Class Distribution: {target_col} ({dataset_name})")
        plt.show()

def plot_missing_heatmap(df, dataset_name):
    """Plots a heatmap of missing values."""
    if df.isnull().sum().sum() > 0:
        plt.figure(figsize=(10, 5))
        sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
        plt.title(f"Missing Values Heatmap: {dataset_name}")
        plt.show()
    else:
        print(f"No missing values found in {dataset_name}!")

# Example Audit: UCI Dropout dataset target is 'Target'
plot_class_distribution(df_dropout, 'Target', 'UCI Dropout Success')
plot_missing_heatmap(df_dropout, 'UCI Dropout Success')

In [ ]:
def plot_class_distribution(df, target_col, dataset_name):
    """Plots the class distribution to check for imbalance."""
    if target_col in df.columns:
        plt.figure(figsize=(8, 4))
        sns.countplot(data=df, x=target_col, hue=target_col, palette="viridis", legend=False)
        plt.title(f"Class Distribution: {target_col} ({dataset_name})")
        plt.show()

def plot_missing_heatmap(df, dataset_name):
    """Plots a heatmap of missing values."""
    if df.isnull().sum().sum() > 0:
        plt.figure(figsize=(10, 5))
        sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
        plt.title(f"Missing Values Heatmap: {dataset_name}")
        plt.show()
    else:
        print(f"No missing values found in {dataset_name}!")

# Example Audit: UCI Dropout dataset target is 'Target'
plot_class_distribution(df_dropout, 'Target', 'UCI Dropout Success')
plot_missing_heatmap(df_dropout, 'UCI Dropout Success')

In [ ]:
# Create a unified list of features
feature_inventory = pd.DataFrame({
    'UCI_Student': pd.Series(df_student_math.columns),
    'UCI_Dropout': pd.Series(df_dropout.columns),
    'Kaggle_Student': pd.Series(df_kaggle.columns)
})

feature_inventory.to_csv(config.PROJECT_ROOT / "paper" / "feature_mapping_worksheet.csv", index=False)
print("Feature mapping worksheet exported to /paper/")